# Telecom Customer Churn Prediction

**Goal:** Predict which customers are likely to cancel their telecom subscription and segment them into actionable risk groups.  
**Dataset:** [Telco Customer Churn — Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)  
**Algorithms:** Decision Tree · XGBoost · K-Means Clustering

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import xgboost as xgb

sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print('Shape:', df.shape)
print('Churn distribution:\n', df['Churn'].value_counts())

churn_rate = df['Churn'].value_counts(normalize=True)['Yes'] * 100
print(f'Churn rate: {churn_rate:.1f}%')

## 3. Data Types Identification

In [ ]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

print('------:Numeric columns:------')
for col in num_cols:
    print(col)

print('\n------:Categorical columns:------')
for col in cat_cols:
    print(col)

## 4. Data Cleaning & Encoding

In [ ]:
# 1. Convert & Impute TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
missing_tc = df['TotalCharges'].isnull().sum()
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
print(f"'TotalCharges' missing values handled: {missing_tc} imputed with median.")

# 2. Preserve 'Churn' and drop 'customerID'
target = df['Churn']
df.drop(['customerID', 'Churn'], axis=1, inplace=True)
print("Dropped 'customerID' column and temporarily removed 'Churn'.")

# 3. One-hot encode low-cardinality categorical variables (≤ 5 unique categories)
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
low_card_cols = [col for col in cat_cols if df[col].nunique() <= 5]

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_array = ohe.fit_transform(df[low_card_cols])
encoded_df = pd.DataFrame(encoded_array, columns=ohe.get_feature_names_out(low_card_cols))

df.drop(columns=low_card_cols, inplace=True)
df_encoded = pd.concat([df.reset_index(drop=True), encoded_df.reset_index(drop=True)], axis=1)

# 4. Add 'Churn' back
df_encoded['Churn'] = target.values
print(f'One-hot encoded: {low_card_cols}')
print(f'Final shape after encoding: {df_encoded.shape}')

In [ ]:
# Check missing values
missing_rate = df_encoded.isnull().sum() / len(df_encoded) * 100
missing_rate = missing_rate[missing_rate > 0].sort_values(ascending=False)
print('Missing Value Rate (%):')
print(missing_rate.round(2) if not missing_rate.empty else 'None — dataset is clean.')

## 5. Exploratory Data Analysis

In [ ]:
# Histogram — Customer Tenure Distribution
plt.figure()
df_encoded['tenure'].hist(bins=30, color='skyblue', edgecolor='black')
plt.title('Customer Tenure Distribution')
plt.xlabel('Tenure (Months)')
plt.ylabel('Number of Customers')
plt.show()
print('Most customers have short tenure (1–12 months). Churn peaks in year one.')

In [ ]:
# Boxplot — Monthly Charges by Churn
plt.figure()
sns.boxplot(
    x='Churn', y='MonthlyCharges', hue='Churn',
    data=df_encoded,
    palette={'Yes': '#fc8d62', 'No': '#66c2a5'},
    legend=False
)
plt.title('Monthly Charges by Churn')
plt.xlabel('Churn')
plt.ylabel('Monthly Charges ($)')
plt.show()
print('Churned customers pay higher monthly charges (higher median and IQR).')

In [ ]:
# Heatmap — Correlation of Numeric Features
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
corr = df_encoded[numeric_cols].corr()

plt.figure()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap of Numeric Features')
plt.show()
print('Tenure & TotalCharges: 0.83 correlation. MonthlyCharges is relatively independent.')

In [ ]:
# Statistical summary
print(df_encoded.groupby('Churn')['MonthlyCharges'].agg(['mean', 'std']))

## 6. Algorithm 1 — Decision Tree

In [ ]:
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn'].map({'No': 0, 'Yes': 1})

# 80% training, 20% testing — stratified to preserve churn ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

cat_cols = X.select_dtypes(include=['object']).columns.tolist()

preproc = ColumnTransformer(
    [('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)],
    remainder='passthrough'
)

pipeline = Pipeline([
    ('prep', preproc),
    ('clf', DecisionTreeClassifier(
        class_weight='balanced',
        max_depth=5,
        random_state=42
    ))
])

# 5-fold cross-validation on training set
scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='f1')
print('5-fold F1 scores:', scores)
print('Mean F1:', scores.mean())

In [ ]:
# Test session
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print('Classification Report:')
print(classification_report(y_test, y_pred))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

## 7. Algorithm 2 — XGBoost

In [ ]:
# Preprocess for XGBoost (requires NumPy arrays via DMatrix)
X_train_t = preproc.fit_transform(X_train)
X_test_t = preproc.transform(X_test)

X_train_np = X_train_t.toarray() if hasattr(X_train_t, 'toarray') else X_train_t
X_test_np = X_test_t.toarray() if hasattr(X_test_t, 'toarray') else X_test_t

dtrain = xgb.DMatrix(X_train_np, label=y_train)
dtest  = xgb.DMatrix(X_test_np,  label=y_test)

# Class imbalance correction
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'scale_pos_weight': scale_pos_weight,
    'max_depth': 4,
    'eta': 0.1,
    'seed': 42
}

model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=200,
    evals=[(dtest, 'val')],
    early_stopping_rounds=10,
    verbose_eval=False
)

y_pred_prob = model.predict(dtest)
y_pred_xgb  = (y_pred_prob > 0.5).astype(int)

print('Classification Report:')
print(classification_report(y_test, y_pred_xgb))
print(f'ROC-AUC: {roc_auc_score(y_test, y_pred_prob):.4f}')

In [ ]:
# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred_xgb)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Yes Churn'],
            yticklabels=['No Churn', 'Yes Churn'])
plt.title('Confusion Matrix — XGBoost Churn Prediction')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'TN={tn}  FP={fp}  FN={fn}  TP={tp}')

## 8. Algorithm 3 — K-Means Customer Segmentation

In [ ]:
num_feats = ['tenure', 'MonthlyCharges', 'TotalCharges']
X_num = df_encoded[num_feats]
X_scaled = StandardScaler().fit_transform(X_num)

# Elbow method
inertias = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure()
plt.plot(range(2, 11), inertias, marker='o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k (K-Means)')
plt.grid(True)
plt.show()

In [ ]:
# Apply K-Means with k=4 (elbow point)
km_final = KMeans(n_clusters=4, random_state=42)
df_encoded['cluster'] = km_final.fit_predict(X_scaled)

print(df_encoded[['tenure', 'MonthlyCharges', 'TotalCharges', 'cluster']].head())

In [ ]:
# PCA visualisation
pca = PCA(n_components=2)
coords = pca.fit_transform(X_scaled)

plt.figure(figsize=(9, 6))
scatter = plt.scatter(coords[:, 0], coords[:, 1],
                      c=df_encoded['cluster'], cmap='tab10', alpha=0.5, s=8)
plt.colorbar(scatter, label='Cluster')
plt.title('Customer Segments (K-Means Clusters via PCA)')
plt.xlabel('pca1')
plt.ylabel('pca2')
plt.tight_layout()
plt.show()

In [ ]:
# Churn rate per cluster
churn_cluster = (
    df_encoded.groupby('cluster')['Churn']
    .value_counts(normalize=True)
    .unstack()
)
print(churn_cluster)

interpretation = {
    0: 'Very loyal, low-risk (safe segment) — ~5% churn',
    1: 'Stable users, low churn risk — ~15% churn',
    2: 'Moderate churn risk — ~25% churn',
    3: 'HIGH RISK: new, high-paying customers — ~48% churn'
}
print()
for k, v in interpretation.items():
    print(f'  Cluster {k}: {v}')

## 9. Results & Business Recommendations

| Model | Accuracy | F1 (Churn) | ROC-AUC |
|-------|----------|------------|---------|
| Decision Tree | 76% | 0.62 | — |
| XGBoost | 76% | 0.63 | **0.84** |

### Key Churn Drivers
- Short tenure (< 12 months)
- Month-to-month contracts
- High monthly charges

### Recommendations
1. **Cluster 3** — Launch targeted retention campaigns with long-term discount incentives
2. **Cluster 2** — Collect proactive feedback before contract renewal dates  
3. **Clusters 0 & 1** — Upsell bundled services to loyal, low-risk customers